# Estimating Population and Demographic Characteristics in Nigeria Using Google Colab

## Seven-Chapter Student Notebook

Choose a Nigerian State/FCT and LGA/Area Council, then progressively analyse mapped settlements, population, sex composition, children under 1 and under 5, detected buildings, and health facilities.

### Where your work is saved
This notebook mounts **your own Google Drive**. Final work is saved permanently under:

`MyDrive/Nigeria_Population_Demography_Course/Student_Work/<State>/<LGA>/`

Temporary processing files may still use `/content`, but your selected boundary and final results are copied to Google Drive so they remain available after the Colab runtime ends.

**Scientific rules:** population values are modelled estimates; GRID3 settlement extents are mapped extents rather than official community boundaries; facility presence does not prove operational status; under-1 is already included in under-5.

Run the notebook from top to bottom.






# Chapter 1 — Choose and map your study LGA

Select a State/FCT and LGA/Area Council from the validated SALB administrative lookup. Stable IDs are `adm1cd` and `adm2cd`.




In [ ]:
# Cell 1 — Install tools, mount Google Drive, and create your personal course folder
!pip -q install geopandas pyogrio shapely pyproj rasterio exactextract requests pyarrow fiona s2sphere ipywidgets folium osm2geojson contextily

from pathlib import Path
import json, re, subprocess, sys, requests
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

# Mount the student's own Google Drive.
drive.mount("/content/drive")

REPO="zubairgis/nigeria-population-demography-course"
RAW=f"https://raw.githubusercontent.com/{REPO}/main"

# Temporary processing area. This may disappear when the Colab runtime resets.
TEMP_ROOT=Path("/content/nigeria_demography_course")
TEMP_ROOT.mkdir(parents=True,exist_ok=True)

# Persistent student area. Files here remain in the student's Google Drive.
DRIVE_ROOT=Path("/content/drive/MyDrive/Nigeria_Population_Demography_Course/Student_Work")
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)

states=pd.read_csv(f"{RAW}/data/lookups/admin_states_source.csv",dtype=str)
lgas=pd.read_csv(f"{RAW}/data/lookups/admin_lgas_source.csv",dtype=str)

assert len(states)==37 and len(lgas)==774
print("Google Drive mounted successfully.")
print("Persistent course folder:", DRIVE_ROOT)
print("State/FCT:",len(states)," LGA/Area Council:",len(lgas))





In [ ]:
# Cell 2 — Choose State/FCT and LGA
state_dropdown=widgets.Dropdown(options=sorted(states.adm1nm.unique()),value="Bayelsa",description="State/FCT:")
lga_dropdown=widgets.Dropdown(description="LGA:")
def refresh(*_):
    opts=sorted(lgas.loc[lgas.adm1nm.eq(state_dropdown.value),"adm2nm"].unique())
    lga_dropdown.options=opts
    lga_dropdown.value="Sagbama" if state_dropdown.value=="Bayelsa" and "Sagbama" in opts else opts[0]
state_dropdown.observe(refresh,names="value"); refresh()
display(widgets.VBox([state_dropdown,lga_dropdown]))
print("Choose above, then run Cell 3.")


In [ ]:
# Cell 3 — Load selected SALB boundary, create your personal work folder, and map it
SELECTED_STATE=state_dropdown.value
SELECTED_LGA=lga_dropdown.value

STATE_ID=states.loc[states.adm1nm.eq(SELECTED_STATE),"adm1cd"].iloc[0]
safe=lambda s: re.sub(r"[^A-Za-z0-9]+","_",str(s)).strip("_")

# Persistent folder for this student's selected study area.
STUDENT_WORK_DIR=DRIVE_ROOT/safe(SELECTED_STATE)/safe(SELECTED_LGA)
STUDENT_WORK_DIR.mkdir(parents=True,exist_ok=True)

# Temporary processing folder for this run.
WORK=TEMP_ROOT/safe(SELECTED_STATE)/safe(SELECTED_LGA)
WORK.mkdir(parents=True,exist_ok=True)

url=f"{RAW}/data/boundaries/source/BNDA_NGA_2000-01-01_lastupdate/by_state/{STATE_ID}_{safe(SELECTED_STATE)}.geojson"
state_gdf=gpd.read_file(url).to_crs(4326)
selected=state_gdf[state_gdf.adm2nm.astype(str).eq(SELECTED_LGA)].copy()

if len(selected)!=1:
    raise ValueError("Selected LGA was not uniquely resolved.")

LGA_ID=str(selected.iloc[0].adm2cd)

# Save the boundary both temporarily and permanently.
LGA_FILE=WORK/f"{LGA_ID}.geojson"
LGA_DRIVE_FILE=STUDENT_WORK_DIR/f"{LGA_ID}_boundary.geojson"
selected.to_file(LGA_FILE,driver="GeoJSON")
selected.to_file(LGA_DRIVE_FILE,driver="GeoJSON")

BBOX=tuple(float(x) for x in selected.total_bounds)
geom=selected.geometry.iloc[0]
center=[geom.centroid.y,geom.centroid.x]

# Download OpenStreetMap roads for the selected LGA.
# Public Overpass servers can occasionally be busy, so try several endpoints.
import osm2geojson
from shapely.geometry import shape

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter",
]

minx,miny,maxx,maxy=BBOX
overpass_query = f"""
[out:json][timeout:120];
(
  way["highway"]({miny},{minx},{maxy},{maxx});
);
out geom;
"""

osm_json = None
last_error = None
for endpoint in OVERPASS_ENDPOINTS:
    try:
        rr = requests.post(
            endpoint,
            data={"data": overpass_query},
            timeout=150,
            headers={"User-Agent":"Hensard-University-Nigeria-Demography-Course/1.0"}
        )
        rr.raise_for_status()
        osm_json = rr.json()
        print("OSM roads source:", endpoint)
        break
    except Exception as e:
        last_error = e
        print("Overpass endpoint unavailable:", endpoint)

if osm_json is None:
    print("OSM roads could not be downloaded; continuing without roads.")
    OSM_ROADS = gpd.GeoDataFrame(
        columns=["highway","name","geometry"], geometry="geometry", crs=4326
    )
else:
    roads_geojson = osm2geojson.json2geojson(osm_json)
    OSM_ROADS = gpd.GeoDataFrame.from_features(roads_geojson["features"], crs=4326)
    if not OSM_ROADS.empty:
        OSM_ROADS = OSM_ROADS[
            OSM_ROADS.geometry.geom_type.isin(["LineString","MultiLineString"])
        ].copy()
        OSM_ROADS = gpd.clip(OSM_ROADS, selected)
        keep_cols=[c for c in ["highway","name","geometry"] if c in OSM_ROADS.columns]
        OSM_ROADS=OSM_ROADS[keep_cols]
    print("OSM road features inside selected LGA:", len(OSM_ROADS))

# Esri satellite basemap only. Settlement names will come from GRID3 in Cell 4.
m=folium.Map(location=center,zoom_start=9,tiles=None,control_scale=True)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Imagery",
    overlay=False,
    control=True
).add_to(m)


folium.GeoJson(
    selected.__geo_interface__,
    name="Selected LGA",
    style_function=lambda feature: {
        "fillColor": "#4da3ff",
        "color": "#0057b8",
        "weight": 3,
        "fillOpacity": 0.18
    },
    tooltip=folium.Tooltip(f"{SELECTED_STATE} — {SELECTED_LGA}")
).add_to(m)

if len(OSM_ROADS)>0:
    folium.GeoJson(
        OSM_ROADS.__geo_interface__,
        name="OSM Roads",
        style_function=lambda feature: {
            "color": "#ffd54f",
            "weight": 2,
            "opacity": 0.9
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[f for f in ["name","highway"] if f in OSM_ROADS.columns],
            aliases=[f"{f.title()}:" for f in ["name","highway"] if f in OSM_ROADS.columns]
        ) if any(f in OSM_ROADS.columns for f in ["name","highway"]) else None
    ).add_to(m)

minx,miny,maxx,maxy=BBOX
m.fit_bounds([[miny,minx],[maxy,maxx]])
folium.LayerControl(collapsed=False).add_to(m)
display(m)

print("State ID:",STATE_ID)
print("LGA ID:",LGA_ID)
print("Persistent student folder:",STUDENT_WORK_DIR)
print("Boundary saved to Google Drive:",LGA_DRIVE_FILE)







# Chapter 2 — Discover settlements and settlement names

Use GRID3 NGA Settlement Extents v3.1 and GRID3 Settlement Names. Zero-name and multiple-name cases remain explicit rather than being forced.




In [ ]:
# Cell 4 — Retrieve settlement extents and use GRID3 Settlement Names cached in GitHub
REPO_DIR=WORK/"repo"
if REPO_DIR.exists():
    subprocess.run(["rm","-rf",str(REPO_DIR)],check=True)

subprocess.run(
    ["git","clone","--depth","1",f"https://github.com/{REPO}.git",str(REPO_DIR)],
    check=True
)

VEC=WORK/"vectors"
VEC.mkdir(exist_ok=True)
bbox=[str(v) for v in BBOX]

# 1. GRID3 Settlement Extents v3.1:
#    downloaded from the versioned GitHub Release configured by the repository.
subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/01_download_sources.py"),
    "--dataset","settlement_extents",
    "--output",str(VEC),
    "--bbox",*bbox
],check=True)

# 2. GRID3 Settlement Names:
#    read ONLY from the copy cached in this GitHub repository.
NAMES_ROOT=REPO_DIR/"data/settlement_names"
NAMES_MANIFEST=NAMES_ROOT/"manifest.json"

if not NAMES_MANIFEST.exists():
    raise FileNotFoundError(
        "The GitHub GRID3 Settlement Names cache is not available yet. "
        "Wait for the 'Cache GRID3 settlement names' GitHub Action to finish, "
        "then restart this notebook from Cell 1."
    )

names_meta=json.loads(NAMES_MANIFEST.read_text())
storage_mode=names_meta["storage_mode"]

if storage_mode=="national":
    names_source=NAMES_ROOT/"GRID3_NGA_settlement_names.geojson"
    grid3_names=gpd.read_file(names_source,bbox=BBOX).to_crs(4326)

elif storage_mode=="by_state":
    parts=[]
    for item in names_meta["files"]:
        p=NAMES_ROOT/item["file"]
        try:
            x=gpd.read_file(p,bbox=BBOX)
            if len(x)>0:
                parts.append(x)
        except Exception:
            pass
    if parts:
        grid3_names=gpd.GeoDataFrame(
            pd.concat(parts,ignore_index=True),
            geometry="geometry",
            crs=parts[0].crs
        ).to_crs(4326)
    else:
        grid3_names=gpd.GeoDataFrame(
            columns=["set_id","set_name","geometry"],
            geometry="geometry",
            crs=4326
        )
else:
    raise ValueError(f"Unknown GRID3 settlement-name storage mode: {storage_mode}")

# Keep only points spatially relevant to the selected LGA.
if len(grid3_names)>0:
    grid3_names=gpd.clip(grid3_names,selected)

GRID3_NAMES_FILE=VEC/"settlement_names.geojson"
grid3_names.to_file(GRID3_NAMES_FILE,driver="GeoJSON")

print("GRID3 settlement-name points from GitHub inside selected LGA:",len(grid3_names))
if len(grid3_names)>0 and "set_name" in grid3_names.columns:
    display(
        grid3_names[
            [c for c in ["set_id","set_name","set_altnam","lganame","geometry"]
             if c in grid3_names.columns]
        ].head(20)
    )

# 3. Match cached GRID3 names to GRID3 v3.1 settlement extents.
SETT=WORK/"settlement_components.gpkg"

subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/04_prepare_settlements.py"),
    "--extents",str(VEC/"settlement_extents.geojson"),
    "--names",str(GRID3_NAMES_FILE),
    "--lga",str(LGA_FILE),
    "--lga-id",LGA_ID,
    "--out",str(SETT)
],check=True)

settlements=gpd.read_file(SETT).to_crs(4326)

print("Settlement components:",len(settlements))
print("Name matching:")
display(settlements.name_match_status.value_counts(dropna=False))

# 4. Map GRID3 settlement names on Esri satellite imagery.
sett_map=folium.Map(location=center,zoom_start=9,tiles=None,control_scale=True)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Imagery",
    overlay=False,
    control=True
).add_to(sett_map)

folium.GeoJson(
    selected.__geo_interface__,
    name="Selected LGA",
    style_function=lambda feature:{
        "fillColor":"#4da3ff",
        "color":"#00b0ff",
        "weight":3,
        "fillOpacity":0.08
    }
).add_to(sett_map)

if len(OSM_ROADS)>0:
    folium.GeoJson(
        OSM_ROADS.__geo_interface__,
        name="OSM Roads",
        style_function=lambda feature:{
            "color":"#ffd54f",
            "weight":2,
            "opacity":0.9
        }
    ).add_to(sett_map)

if len(grid3_names)>0:
    for _,r in grid3_names.iterrows():
        nm=str(r.get("set_name","")).strip()
        if nm:
            folium.Marker(
                [r.geometry.y,r.geometry.x],
                icon=folium.DivIcon(
                    html=f'<div style="font-size:10px;font-weight:bold;color:white;'
                         f'text-shadow:1px 1px 2px black;white-space:nowrap;">{nm}</div>'
                )
            ).add_to(sett_map)

minx,miny,maxx,maxy=BBOX
sett_map.fit_bounds([[miny,minx],[maxy,maxx]])
folium.LayerControl(collapsed=False).add_to(sett_map)
display(sett_map)



# Chapter 3 — Estimate total population\n\nUse the **WorldPop v2 API** for the selected LGA. No WorldPop GeoTIFF is downloaded or stored.\n



In [ ]:
# Cell 5 — Set up the WorldPop v2 API and choose an indicator
import time
from shapely.geometry import mapping

WORLDPOP_API = "https://api.worldpop.org/v2"
WORLDPOP_YEAR = 2025
WORLDPOP_RESOLUTION = "100m"

# WorldPop v2 accepts Polygon or MultiPolygon geometry in WGS84.
LGA_GEOMETRY = mapping(selected.to_crs(4326).geometry.iloc[0])

def worldpop_submit(endpoint, payload, poll_seconds=2, timeout_seconds=180):
    """Submit a WorldPop v2 task and wait for the result."""
    r = requests.post(f"{WORLDPOP_API}/{endpoint}", json=payload, timeout=60)
    r.raise_for_status()
    submitted = r.json()
    task_id = submitted.get("task_id")
    if not task_id:
        raise RuntimeError(f"WorldPop did not return a task_id: {submitted}")

    started = time.time()
    while True:
        check = requests.get(f"{WORLDPOP_API}/tasks/{task_id}", timeout=60)
        check.raise_for_status()
        info = check.json()

        status = str(info.get("status","")).lower()
        if status == "success":
            return info["result"]
        if status in {"failure","failed","error"}:
            raise RuntimeError(f"WorldPop task failed: {info.get('error') or info}")

        if time.time() - started > timeout_seconds:
            raise TimeoutError(f"WorldPop task {task_id} exceeded {timeout_seconds} seconds.")
        time.sleep(poll_seconds)

def query_population(geometry):
    return worldpop_submit("population", {
        "geojson": geometry,
        "year": WORLDPOP_YEAR,
        "resolution": WORLDPOP_RESOLUTION,
    })

def query_agesex(geometry, age_range=(0,90), sex="both"):
    return worldpop_submit("agesex", {
        "geojson": geometry,
        "year": WORLDPOP_YEAR,
        "age_range": list(age_range),
        "sex": sex,
        "resolution": WORLDPOP_RESOLUTION,
    })

indicator_dropdown = widgets.Dropdown(
    options=[
        ("Total population","total"),
        ("Male population","male"),
        ("Female population","female"),
        ("Children under 1","u1"),
        ("Children under 5","u5"),
    ],
    value="total",
    description="Indicator:"
)
display(indicator_dropdown)

print("WorldPop API configured for:", WORLDPOP_YEAR, WORLDPOP_RESOLUTION)
print("No raster file will be downloaded or stored.")


In [ ]:
# Cell 6 — Get total population for the selected LGA from WorldPop API
pop_result = query_population(LGA_GEOMETRY)

population_total_est = float(pop_result["total_population"])

print(f"{SELECTED_LGA}, {SELECTED_STATE}")
print(f"Estimated total population ({WORLDPOP_YEAR}): {population_total_est:,.0f}")
print("WorldPop source:", pop_result.get("data_source"))
print("Area (km²):", pop_result.get("area_km2"))

# Start the small LGA summary table used by later chapters.
lga_pop = pd.DataFrame([{
    "adm2cd": LGA_ID,
    "population_total_est": population_total_est,
    "population_reference_year": WORLDPOP_YEAR,
    "population_dataset_id": "WorldPop Global 2 via API v2",
    "resolution": WORLDPOP_RESOLUTION,
}])
display(lga_pop)


# Chapter 4 — Describe male and female population\n\nUse the WorldPop age-sex API to obtain the male and female population for the selected LGA.\n



In [ ]:
# Cell 7 — Get male and female population from WorldPop API
agesex_all = query_agesex(LGA_GEOMETRY, age_range=(0,90), sex="both")
pyramid = pd.DataFrame(agesex_all["agesex_pyramid"])

population_male_est = float(pd.to_numeric(pyramid["male"], errors="coerce").sum())
population_female_est = float(pd.to_numeric(pyramid["female"], errors="coerce").sum())

lga_pop["population_male_est"] = population_male_est
lga_pop["population_female_est"] = population_female_est

sex = pd.DataFrame({
    "group":["Male","Female"],
    "estimate":[population_male_est,population_female_est]
})
display(sex)

ax=sex.plot.bar(
    x="group", y="estimate", legend=False,
    title=f"{SELECTED_LGA}: modelled male and female population"
)
ax.set_ylabel("Estimated people")

print("Male + female:", f"{population_male_est + population_female_est:,.0f}")
print("API total:", f"{population_total_est:,.0f}")


# Chapter 5 — Identify children under 1 and under 5\n\nUse the WorldPop age-sex API for ages 0–4. Under-1 is included within under-5. An optional settlement explorer lets students query one mapped settlement at a time without downloading rasters.\n



In [ ]:
# Cell 8 — Get children under 1 and under 5 from WorldPop API
# Request only ages 0–4. The returned classes include age 0 and ages 1–4.
agesex_u5 = query_agesex(LGA_GEOMETRY, age_range=(0,4), sex="both")
u5_pyramid = pd.DataFrame(agesex_u5["agesex_pyramid"])

# WorldPop classes are normally "00" for age 0–1 and "01" for ages 1–5.
u5_pyramid["class"] = u5_pyramid["class"].astype(str).str.zfill(2)
u5_pyramid["both"] = (
    pd.to_numeric(u5_pyramid["male"], errors="coerce").fillna(0)
    + pd.to_numeric(u5_pyramid["female"], errors="coerce").fillna(0)
)

u1_rows = u5_pyramid[u5_pyramid["class"].eq("00")]
u1 = float(u1_rows["both"].sum())
u5 = float(u5_pyramid["both"].sum())

if not (0 <= u1 <= u5 <= population_total_est * 1.01):
    raise ValueError("Unexpected demographic hierarchy returned by the API.")

lga_pop["population_u1_est"] = u1
lga_pop["population_u5_est"] = u5

child_summary = pd.DataFrame({
    "indicator":["Under 1","Under 5"],
    "estimate":[u1,u5],
    "percent_total":[100*u1/population_total_est,100*u5/population_total_est]
})
display(child_summary.round(2))
display(u5_pyramid)

print("Under-1 is included within under-5; do not add it again.")


In [ ]:
# Cell 9 — Optional API explorer: choose one settlement and one indicator
# This intentionally queries ONE settlement at a time to avoid hundreds of API jobs.

settlement_options = []
for i,row in settlements.reset_index(drop=True).iterrows():
    name = row.get("settlement_name")
    if pd.isna(name) or str(name).strip()=="":
        label = f"Unnamed settlement component {i+1}"
    else:
        label = str(name)
    settlement_options.append((f"{label} [{row['settlement_component_id']}]", row["settlement_component_id"]))

settlement_dropdown = widgets.Dropdown(
    options=settlement_options,
    description="Settlement:",
    layout=widgets.Layout(width="95%")
)
sett_indicator_dropdown = widgets.Dropdown(
    options=[
        ("Total population","total"),
        ("Male population","male"),
        ("Female population","female"),
        ("Children under 1","u1"),
        ("Children under 5","u5"),
    ],
    value=indicator_dropdown.value,
    description="Indicator:"
)

display(widgets.VBox([settlement_dropdown,sett_indicator_dropdown]))
print("Choose a settlement and indicator above, then run Cell 10.")


In [ ]:
# Cell 10 — Query the selected settlement/indicator
chosen_id = settlement_dropdown.value
chosen_indicator = sett_indicator_dropdown.value
chosen = settlements[settlements["settlement_component_id"].astype(str).eq(str(chosen_id))].copy()

if len(chosen) != 1:
    raise ValueError("Selected settlement component was not uniquely resolved.")

chosen_geom = mapping(chosen.to_crs(4326).geometry.iloc[0])

if chosen_indicator == "total":
    result = query_population(chosen_geom)
    value = float(result["total_population"])

else:
    if chosen_indicator in {"u1","u5"}:
        result = query_agesex(chosen_geom, age_range=(0,4), sex="both")
    else:
        result = query_agesex(chosen_geom, age_range=(0,90), sex="both")

    p = pd.DataFrame(result["agesex_pyramid"])
    p["class"] = p["class"].astype(str).str.zfill(2)
    male = pd.to_numeric(p["male"], errors="coerce").fillna(0)
    female = pd.to_numeric(p["female"], errors="coerce").fillna(0)

    if chosen_indicator == "male":
        value = float(male.sum())
    elif chosen_indicator == "female":
        value = float(female.sum())
    elif chosen_indicator == "u1":
        value = float((male + female)[p["class"].eq("00")].sum())
    elif chosen_indicator == "u5":
        value = float((male + female).sum())

settlement_api_result = pd.DataFrame([{
    "settlement_component_id": chosen_id,
    "settlement_name": chosen.iloc[0].get("settlement_name"),
    "indicator": chosen_indicator,
    "estimate": value,
    "reference_year": WORLDPOP_YEAR,
    "source": "WorldPop API v2"
}])

display(settlement_api_result)
print("No raster was downloaded.")


# Chapter 6 — Locate health facilities and explore the road network

Use GRID3/NHFR health-facility records and OpenStreetMap roads for the selected LGA. Building-count analysis has been removed from the student workflow.




In [ ]:
# Cell 8 — Prepare health facilities inside the selected LGA
subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/01_download_sources.py"),
    "--dataset","health_v2",
    "--output",str(VEC),
    "--bbox",*bbox
],check=True)

HEALTH=WORK/"health.gpkg"

subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/06_prepare_health_facilities.py"),
    "--facilities",str(VEC/"health_v2.geojson"),
    "--lga",str(LGA_FILE),
    "--out",str(HEALTH)
],check=True)

health=gpd.read_file(HEALTH).to_crs(4326)

print("Health facilities inside selected LGA:",len(health))

if len(health)>0:
    display(
        health[
            [c for c in [
                "facility_name",
                "ownership",
                "facility_level",
                "latitude",
                "longitude"
            ] if c in health.columns]
        ].head(20)
    )



# Chapter 7 — Reconcile, interpret and export

Build final LGA and settlement summaries, run consistency checks, and export CSV tables and PNG maps.




In [ ]:
# Cell 11 — Final tables and demographic consistency checks

# Settlement table contains settlement identity and naming information only.
sett_table = settlements.drop(columns="geometry").copy()

# LGA demographic checks.
if not (
    0 <= float(lga_pop.loc[0,"population_u1_est"])
    <= float(lga_pop.loc[0,"population_u5_est"])
    <= float(lga_pop.loc[0,"population_total_est"]) * 1.01
):
    raise ValueError("Demographic hierarchy check failed.")

sex_sum = (
    float(lga_pop.loc[0,"population_male_est"])
    + float(lga_pop.loc[0,"population_female_est"])
)

consistency = pd.DataFrame([{
    "total_population_api": float(lga_pop.loc[0,"population_total_est"]),
    "male_plus_female": sex_sum,
    "difference": sex_sum - float(lga_pop.loc[0,"population_total_est"]),
    "under_1": float(lga_pop.loc[0,"population_u1_est"]),
    "under_5": float(lga_pop.loc[0,"population_u5_est"]),
}])

display(consistency.round(2))

show_cols = [
    c for c in [
        "settlement_component_id",
        "settlement_name",
        "name_match_status"
    ]
    if c in sett_table.columns
]

display(sett_table[show_cols].head(20))



In [ ]:
# Cell 12 — Export ONLY CSV tables and PNG map files to Google Drive
EXPORT = STUDENT_WORK_DIR / "Final_Results"
EXPORT.mkdir(parents=True, exist_ok=True)

# Remove older files that are not CSV or PNG.
for old in EXPORT.iterdir():
    if old.is_file() and old.suffix.lower() not in {".csv", ".png"}:
        old.unlink()

# ----------------------------
# CSV TABLES
# ----------------------------
lga_pop.to_csv(EXPORT / "lga_population_summary.csv", index=False)
sett_table.to_csv(EXPORT / "settlement_summary.csv", index=False)
health.drop(columns="geometry").to_csv(
    EXPORT / "health_facilities.csv", index=False
)

if "settlement_api_result" in globals():
    settlement_api_result.to_csv(
        EXPORT / "selected_settlement_population_query.csv",
        index=False
    )

if len(OSM_ROADS) > 0:
    roads_csv = OSM_ROADS.drop(columns="geometry").copy()
    roads_csv["geometry_wkt"] = OSM_ROADS.geometry.to_wkt()
    roads_csv.to_csv(EXPORT / "osm_roads.csv", index=False)

# ----------------------------
# PNG MAPS
# ----------------------------
import matplotlib.pyplot as plt
import contextily as ctx

selected_3857 = selected.to_crs(epsg=3857)
settlements_3857 = settlements.to_crs(epsg=3857)

if len(health) > 0:
    health_3857 = health.to_crs(epsg=3857)
else:
    health_3857 = health.copy()

if len(OSM_ROADS) > 0:
    roads_3857 = OSM_ROADS.to_crs(epsg=3857)
else:
    roads_3857 = OSM_ROADS.copy()

if "grid3_names" in globals() and len(grid3_names) > 0:
    grid3_names_3857 = grid3_names.to_crs(epsg=3857)
else:
    grid3_names_3857 = gpd.GeoDataFrame(
        columns=["set_name","geometry"], geometry="geometry", crs="EPSG:3857"
    )

xmin, ymin, xmax, ymax = selected_3857.total_bounds
pad_x = max((xmax - xmin) * 0.05, 100)
pad_y = max((ymax - ymin) * 0.05, 100)

# -----------------------------------
# MAP 1 — Study area + OSM roads
# -----------------------------------
fig, ax = plt.subplots(figsize=(10, 10))

selected_3857.boundary.plot(
    ax=ax,
    linewidth=2.5
)

if len(roads_3857) > 0:
    roads_3857.plot(
        ax=ax,
        linewidth=1.0
    )

# GRID3 settlement labels on Map 1
if len(grid3_names_3857) > 0:
    for _,r in grid3_names_3857.iterrows():
        nm=str(r.get("set_name","")).strip()
        if nm:
            ax.text(
                r.geometry.x, r.geometry.y, nm,
                fontsize=5.5, ha="center", va="center",
                bbox=dict(facecolor="white", alpha=0.55, edgecolor="none", pad=0.4)
            )

ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    attribution=False
)

# Note: xyzservices in Colab does not consistently expose an Esri
# labels/reference provider. OSM roads and vector layers provide the
# contextual labels/features on top of the satellite imagery.


ax.set_xlim(xmin - pad_x, xmax + pad_x)
ax.set_ylim(ymin - pad_y, ymax + pad_y)
ax.set_title(
    f"Study Area and OSM Roads\n{SELECTED_LGA}, {SELECTED_STATE}",
    fontsize=14
)
ax.set_axis_off()

study_map_png = EXPORT / "01_study_area_roads_map.png"
plt.savefig(
    study_map_png,
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)

# -----------------------------------
# MAP 2 — Settlements + health + roads
# -----------------------------------
fig, ax = plt.subplots(figsize=(10, 10))

selected_3857.boundary.plot(
    ax=ax,
    linewidth=2.5
)

if len(settlements_3857) > 0:
    settlements_3857.plot(
        ax=ax,
        alpha=0.35,
        edgecolor="orange",
        linewidth=0.5
    )

if len(roads_3857) > 0:
    roads_3857.plot(
        ax=ax,
        linewidth=1.0
    )

# GRID3 settlement labels on Map 2
if len(grid3_names_3857) > 0:
    for _,r in grid3_names_3857.iterrows():
        nm=str(r.get("set_name","")).strip()
        if nm:
            ax.text(
                r.geometry.x, r.geometry.y, nm,
                fontsize=5.5, ha="center", va="center",
                bbox=dict(facecolor="white", alpha=0.55, edgecolor="none", pad=0.4)
            )

if len(health_3857) > 0:
    health_3857.plot(
        ax=ax,
        markersize=20
    )

ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    attribution=False
)

# Note: xyzservices in Colab does not consistently expose an Esri
# labels/reference provider. OSM roads and vector layers provide the
# contextual labels/features on top of the satellite imagery.


ax.set_xlim(xmin - pad_x, xmax + pad_x)
ax.set_ylim(ymin - pad_y, ymax + pad_y)
ax.set_title(
    f"Settlements, Health Facilities, and OSM Roads\n{SELECTED_LGA}, {SELECTED_STATE}",
    fontsize=14
)
ax.set_axis_off()

result_map_png = EXPORT / "02_settlements_health_roads_map.png"
plt.savefig(
    result_map_png,
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)

print("Export complete.")
print("Only CSV and PNG files are stored in:")
print(EXPORT)

# ----------------------------
# ZIP — CSV + PNG ONLY
# ----------------------------
ZIP_DRIVE = STUDENT_WORK_DIR / (
    f"{safe(SELECTED_STATE)}_{safe(SELECTED_LGA)}_maps_csv.zip"
)

if ZIP_DRIVE.exists():
    ZIP_DRIVE.unlink()

files_to_zip = [
    p.name for p in EXPORT.iterdir()
    if p.suffix.lower() in {".csv", ".png"}
]

subprocess.run(
    ["zip", "-q", str(ZIP_DRIVE), *files_to_zip],
    cwd=EXPORT,
    check=True
)

print("ZIP containing PNG maps + CSV only:")
print(ZIP_DRIVE)

from google.colab import files
files.download(str(ZIP_DRIVE))






## Completed

You have selected an LGA, mapped settlements, estimated population and demographics, added health-facility and road context, reconciled the results, and exported the outputs.


